# AI Incident Response — Dev Log

## Objetivo e papel no pipeline

`core/incident_response` é o registro real de incidentes operacionais do
AthenaGov AI, com runbook declarativo de resposta por severidade
(`runbooks.yaml`) e ciclo de vida `open -> investigating -> resolved`.

Este dev-log demonstra o caso de uso mais honesto possível: registrar,
investigar e resolver o **achado real do `red_team_lab`** (gap de detecção
do `prompt_security` em português) como um incidente de verdade.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import tempfile
from pathlib import Path
from core.incident_response.log import IncidentLog, get_runbook
from shared.schemas import IncidentStatus, RiskLevel

demo_dir = Path(tempfile.mkdtemp(prefix="incident_demo_"))
log = IncidentLog(storage_path=demo_dir / "incidents.json")

incident = log.report_incident(
    title="Gap de detecção do prompt_security explorado em teste de red-team",
    description="run_red_team_suite() encontrou que RT-01 (injeção direta em PT) não é detectada pelo motor atual.",
    severity=RiskLevel.HIGH,
)
print(f"Incidente criado: {incident.incident_id[:8]}... | status={incident.status.value}")

log.update_status(incident.incident_id, IncidentStatus.INVESTIGATING)
resolved = log.update_status(
    incident.incident_id, IncidentStatus.RESOLVED,
    resolution_notes="Adicionado ao backlog de melhoria do prompt_security (ver red_team_lab/CHANGELOG.md).",
)
print(f"Incidente resolvido: status={resolved.status.value}, resolvido em {resolved.resolved_at}")
print()
print("Runbook para severidade HIGH:")
for step in get_runbook(RiskLevel.HIGH):
    print(f"  - {step}")

Incidente criado: 39bca5c6... | status=open
Incidente resolvido: status=resolved, resolvido em 2026-08-20 23:50:57.915747+00:00

Runbook para severidade HIGH:
  - Conter: restringir o escopo do sistema afetado (ex. desativar apenas a funcionalidade envolvida).
  - Preservar evidências relevantes em audit_logs.
  - Avaliar necessidade de notificação (LGPD Art. 48) caso a caso.
  - Corrigir e validar com testes antes de reativar.
  - Registrar no post-mortem, mesmo que não obrigatório.


## Rodando a suíte de testes

```
"C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe" -m pytest core/incident_response/tests -v
```

12 testes contra arquivo temporário real: criação, persistência entre
instâncias, transições de status, exigência de notas na resolução,
proibição de reabertura, filtragem por status, `related_event_ids`, runbook
por severidade (inclusive verificação de que o runbook CRITICAL menciona a
notificação do Art. 48 da LGPD).

## Handoff Summary

- **Status:** ✅ done — 12/12 testes passando.
- **TODO onda futura:** `red_team_lab`/`blockchain_audit_layer` abrindo
  incidente automaticamente quando encontram um gap/falha, hoje é manual
  (`report_incident` chamado explicitamente).